In [13]:
import numpy as np
from tensorflow.keras.datasets import mnist
from sklearn.decomposition import PCA
import scipy.linalg

# Function to load and preprocess MNIST data
def load_mnist_data():
    (train_images, train_labels), _ = mnist.load_data()
    # Flatten the images and normalize
    train_images = train_images.reshape((train_images.shape[0], -1))
    train_images = train_images.astype('float32') / 255
    return train_images

# Given a unit vector x, compute the orthogonal complement of x
def orthogonal_complement(x):
    u, s, vh = np.linalg.svd(x[:, np.newaxis].T)
    return vh[1:].T

# Function to align vectors for the coreset construction
def align_vectors(u, v):
    # Normalize u and v
    u_normalized = u / np.linalg.norm(u)
    v_normalized = v / np.linalg.norm(v)

    # Cross product to find rotation axis
    axis = np.cross(u_normalized, v_normalized)
    sin_angle = np.linalg.norm(axis)
    cos_angle = np.dot(u_normalized, v_normalized)
    axis = axis / sin_angle

    # Skew-symmetric cross-product matrix
    K = np.array([[0, -axis[2], axis[1]],
                  [axis[2], 0, -axis[0]],
                  [-axis[1], axis[0], 0]])

    # Rotation matrix
    I = np.eye(3)
    R = I + K + K @ K * ((1 - cos_angle) / (sin_angle ** 2))
    return R

# Function to compute the one segment coreset for MNIST
def     one_segment(P):
    # Assume all weights are equal for simplicity
    W = np.ones(P.shape[0])
    sqrt_W = np.sqrt(W)
    
    # Embed P with an additional 1s column to handle affine transformations
    P_extended = np.hstack([np.ones((P.shape[0], 1)), P])
    
    # Weighted PCA via SVD
    X_weighted = P_extended * sqrt_W[:, np.newaxis]
    U, D, Vt = np.linalg.svd(X_weighted, full_matrices=False)
    
    # Get the first principal component
    first_pc = Vt.T[:, 0]
    scaling_factor = np.linalg.norm(first_pc)
    
    # Normalize and prepare output
    C = first_pc.reshape(1, -1) / scaling_factor
    return C, scaling_factor

# Main function to load data, compute coreset, and test
def main():
    data = load_mnist_data()
    coreset, scale = one_segment(data[:10000])  # Use a subset for quick testing
    
    print("Coreset shape:", coreset.shape)
    print("Scale factor:", scale)

if __name__ == '__main__':
    main()


Coreset shape: (1, 785)
Scale factor: 0.9999999999999999
